<a href="https://colab.research.google.com/github/GokulKrishna2017/LangChain/blob/main/Day_3_Parallel_Chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Dependencies

In [ ]:
!pip install -q \
langchain \
langchain-community \
transformers \
accelerate \
bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


###Imports

In [ ]:
from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

from langchain_community.llms import HuggingFacePipeline

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

/tmp/ipykernel_1953/3970539075.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


###Configuring Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb__4bit_use_double_quant=True
)

Loading the Model

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model =  AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto"
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Creating HuggingFace Pipeline

In [ ]:
pipe = pipeline(
    task = "text-generation",
    model = model,
    tokenizer = tokenizer
)

##Converting Pipeline to LangChain LLM

In [ ]:
llm = HuggingFacePipeline(
    pipeline = pipe
)

/tmp/ipykernel_1953/2360307418.py:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(


##Promt
###Notes Prompt

In [ ]:
prompt1 = PromptTemplate(
    template= "Generate short and simple notes from the following text. \n\n {text}",
    input_variables = ["text"]
)

###Quiz Prompt

In [ ]:
prompt2 = PromptTemplate(
    template = "Generate 5 short question- answer pairs from the following text.\n\n{text}",
    input_variables= ["text"]
)

Merging the prompt

In [ ]:
prompt3 = PromptTemplate(
    template="create a clean study guide.\n\n Notes:\n{notes}\n\nQuiz:\n{quiz}",
    input_variables= ["notes","quiz"]
)

##create Output Parser

In [ ]:
parser = StrOutputParser()

Creating parallel chain

In [ ]:
parallel_chain = RunnableParallel(
    {
        "notes": prompt1 | llm | parser,
        "quiz": prompt2 | llm | parser
    }
)

##Creating Merge chain

In [ ]:
merge_chain = prompt3 | llm | parser

Combining Everything

In [ ]:
chain = parallel_chain | merge_chain

##Text

In [ ]:
text = """
Support vector machines (SVMs) are a set of supervised learning methods used
for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function
(called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified.
"""

##Running the chain

In [ ]:
result = chain.invoke(
    {"text": text}
    )

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces

In [ ]:
print(result)

create a clean study guide.

 Notes:
Generate short and simple notes from the following text. 

 
Support vector machines (SVMs) are a set of supervised learning methods used
for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function
(called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified.
Kernel functions transform input data into higher dimensional feature space.
SVM with linear kernel simply performs a linear separation in this space.

SVM has been successfully applied to various fields such as bioinformatics,
computer vision, natural language processing etc.

1. Support Vector Machines (SVMs)
2. Supervised Learning Methods  
3. Classification, Regression, Outliers Detection

Advantages:
4. Effec